### Middleware:
A way to more tightly control what happens inside the agent. It is useful for following:
 - Tracking agent behavior with logging, analytics and debugging.
 - Transforming prompts, tools selection and output formatting.
 - Adding retries, fallbacks, amd early termination logic. 
 - Applying rate limites, guard rails, and PII detection.

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization:
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. 
 Summarization is useful for the following:
 - long running conversations that exceeded context window.
 - multi turn dialogues with extensive history.
 - Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.checkpoints import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

#message based summarization:
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger=("message_count", 10),
            keep = ("message_count", 4)
        )
    ]
)


#run with the thread id:
config = {"configurable": {"thread_id": "12345"}}

#alternate test data:
questions = [
    "What is the capital of France?",
    "What is the largest mammal?",
    "Who wrote 'To Kill a Mockingbird'?",
    "What is the speed of light?",
    "Who painted the Mona Lisa?",
    "What is the tallest mountain in the world?",
    "Who is the current president of the United States?",
    "What is the chemical symbol for gold?",
    "Who discovered penicillin?",
    "What is 2+2?"
]

for q in questions:
    response = agent.invoke({"message":[HumanMessage(content=q)]}, config=config)
    print(f"messages:{response}")
    print(f"messages:{len(response['messages'])}")



In [ ]:
#Based on Token size:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.checkpoints import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels- return long response to use more tokens."""
    return f"""hotels in {city}:
    1. Hotel A - 5 stars, $200/night
    2. Hotel B - 4 stars, $150/night
    3. Hotel C - 3 stars, $100/night"""

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger=("token_count", 500),
            keep = ("token_count", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "12345"}}

#token counter(approximate) for the questions:
def count_tokens(messages):
    total_chars = sum(len(str(msg.content)) for msg in messages)
    return total_chars // 4  # Approximate token count

#run test:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix","Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"]

for city in cities:
    response = agent.invoke(
        {"message":[HumanMessage(content=f"Search hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response['messages'])
    print(f"city: ~{tokens} tokens, {len(response['messages'])}")
    print(f"messages: {response['messages']}")
    
    



In [ ]:
#Based on fraction:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.checkpoints import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels- return long response to use more tokens."""
    return f"""hotels in {city}:
    1. Hotel A - 5 stars, $200/night
    2. Hotel B - 4 stars, $150/night
    3. Hotel C - 3 stars, $100/night"""

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger=("fraction", 0.005 ),  # Summarize when the conversation reaches 0.5% =~640 tokensof the model's context window
            keep = ("fraction", 0.002),  # Summarize when the conversation reaches 0.2% =~256 tokensof the model's context window
        )
    ]
)

config = {"configurable": {"thread_id": "12345"}}

#token counter(approximate) for the questions:
def count_tokens(messages):
    total_chars = sum(len(str(msg.content)) for msg in messages)
    return total_chars // 4  # Approximate token count

#run test:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix","Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"]

for city in cities:
    response = agent.invoke(
        {"message":[HumanMessage(content=f"Search hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response['messages'])
    fraction = tokens / 128000  # Assuming a context window of 128k tokens
    print(f"city: ~{tokens} tokens ({fraction:0.4%}) {len(response['messages'])}")
    print(f"messages: {response['messages']}")
    
    


### Human in the loop Middleware:
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. It is useful for the following:
 - High-stakes operations require human approval(eg: database writes, financial transactions)
 - Compliance workflows where human oversight is mandatory
 - Long-running conversations where human feedback guides the agents.

In [ ]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import  InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email content based on email ID."""
    return f"Email ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve","edit", "reject"],
                },
                "read_email_tool": False,  # Interrupt immediately without any decision options
            }
        )
    ]
)

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "test-approve"}}

#step1 : Request to read email content:
response = agent.invoke(
    {"message": [HumanMessage(content="Send email to john@test.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am.")]},
    config=config
)

#Step2: Approve the email sending action:
if "__interrupt__" in response:
    print("|| Paused! Approving the email sending action... ||")

    response = agent.invoke(
        Command(
            resume={
                "decision": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Response after approval: {response['messages'][-1].content}")
   


### Reject:


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import  InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email content based on email ID."""
    return f"Email ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve","edit", "reject"],
                },
                "read_email_tool": False,  # Interrupt immediately without any decision options
            }
        )
    ]
)

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "test-reject"}}

#step1 : Request to read email content:
response = agent.invoke(
    {"message": [HumanMessage(content="Send email to john@test.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am.")]},
    config=config
)

#Step2: Approve the email sending action:
if "__interrupt__" in response:
    print("|| Paused! Approving the email sending action... ||")

    response = agent.invoke(
        Command(
            resume={
                "decision": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Response after approval: {response['messages'][-1].content}")
   
